# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nhlalenhle/FlyRank-Nhlalenhle/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the Week-5 **content-refresh prioritisation** model as an ML engineer would audit a research claim.

The audit has four goals:

1. ask two constructive methodology questions about findings in the FlyRank research report;
2. compare the Week-5 model with a stricter grouped validation design;
3. check every model feature for outcome leakage;
4. rewrite claims so they stay within the evidence.

The model is decision-support. `went_dark` is a measured proxy outcome, not proof that a page needs a refresh or that a refresh would cause recovery.

## 1. Two paper findings + my methodology questions

I selected two findings because they are close to my own modeling lane: **Finding 4 — The Freshness Multiplier** and **Finding 8 — The Age × Freshness Matrix**.

### Finding 4 — The Freshness Multiplier

The report observes a strong growth-to-decline ratio in some freshness windows and separately reports a large difference between a recently refreshed 365+ cohort and an older stale cohort. The report also explicitly notes that the 361+ freshness bucket is small and unstable.

**Methodology question:** what is the exact outcome/label behind “growth” and “decline,” and are the refreshed and stale cohorts comparable on their pre-refresh performance, age, topic mix, and other characteristics? The report is appropriately cautious about small cells, but a reader would still benefit from knowing whether the comparison is descriptive only or whether a design was used to control for pre-existing differences. A before/after or matched-cohort analysis would answer a different causal question from a cross-sectional comparison.

### Finding 8 — The Age × Freshness Matrix

The report compares health scores across age and freshness cells and identifies “Growth Engine,” “Refresh Winners,” “Decay Zone,” and “Survivors.”

**Methodology question:** how is the update/freshness label generated, and can the label itself contain information that is also reflected in the outcome or health score? The report says updates are detected through internal workflow logic and algorithmic signals, including body edits, metadata, links, and UX changes. It would be useful to know the exact observation date for the update signal and to ensure the health score being compared is measured after the update without using post-outcome information in the group definition.

These are methodology questions, not judgments about the report. The report itself discloses several relevant limitations, including confounding by content age, the use of descriptive comparisons for headline findings, and the fact that the study identifies patterns rather than causal effects. citeturn0search0

In [ ]:
# Reproducibility setup
import os
import numpy as np
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "https://raw.githubusercontent.com/Nhlalenhle/FlyRank-Nhlalenhle/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)
print("Columns:", list(df.columns))

## 2. My model under an honest split (before/after)

The Week-5 notebook already used a grouped split by `client_hash_id`. That is stronger than a random row split because pages from one client cannot appear in both training and test sets.

For this audit, I make the validation design explicit and compare it with a **row-random split** as a sensitivity check. The random split is the “before” reference; the grouped split is the “after” honest estimate.

The model, features, target, and metric stay fixed. Only the validation design changes.

The primary metric remains **Precision@20**, because the operational use is a small ranked review queue.

In [ ]:
required = [
    "client_hash_id",
    "feb_impressions", "feb_clicks", "feb_ctr", "feb_position",
    "content_age_days",
    "march_clicks"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected Week-5 columns: {missing}")

audit_df = df[required].dropna(subset=["client_hash_id", "march_clicks"]).copy()
audit_df["went_dark"] = (audit_df["march_clicks"] == 0).astype(int)

feature_cols = [
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_position",
    "content_age_days",
]

X = audit_df[feature_cols].copy()
y = audit_df["went_dark"].copy()
groups = audit_df["client_hash_id"].copy()

print("Measured rows:", len(audit_df))
print("Went-dark rate:", f"{y.mean():.1%}")
print("Clients:", groups.nunique())

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, score, k=20):
    y_true = np.asarray(y_true)
    score = np.asarray(score)
    k = min(k, len(y_true))
    order = np.argsort(-score, kind="mergesort")[:k]
    return float(y_true[order].mean())

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=2000, random_state=42))
    ])

def baseline_score(X_part):
    return (
        2 * (X_part["content_age_days"] >= 365).astype(int)
        + 2 * (X_part["feb_ctr"] < 0.01).astype(int)
        + 1 * (X_part["feb_position"] >= 10).astype(int)
    )

# BEFORE: random row split.
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
random_model = make_model()
random_model.fit(X_tr_r, y_tr_r)
random_proba = random_model.predict_proba(X_te_r)[:, 1]

random_base = baseline_score(X_te_r)

# AFTER: grouped split by client.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]

grouped_model = make_model()
grouped_model.fit(X_tr_g, y_tr_g)
grouped_proba = grouped_model.predict_proba(X_te_g)[:, 1]

grouped_base = baseline_score(X_te_g)

comparison = pd.DataFrame({
    "Validation": ["Before: random rows", "After: grouped by client"],
    "Model Precision@20": [
        precision_at_k(y_te_r, random_proba, 20),
        precision_at_k(y_te_g, grouped_proba, 20)
    ],
    "Baseline Precision@20": [
        precision_at_k(y_te_r, random_base, 20),
        precision_at_k(y_te_g, grouped_base, 20)
    ],
    "Model ROC-AUC": [
        roc_auc_score(y_te_r, random_proba),
        roc_auc_score(y_te_g, grouped_proba)
    ],
    "Test rows": [len(y_te_r), len(y_te_g)],
    "Test clients": ["n/a", groups.iloc[test_idx].nunique()]
})

display(comparison.round(3))

print(
    "Grouped client overlap:",
    len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
)

### What changed?

The important comparison is not simply which number is larger. The grouped result answers a harder generalisation question: **how does the model perform when the client identity is not shared between training and test?**

If the grouped score is lower than the random-row score, that is evidence that the random split was optimistic for this use case. If the scores are similar, that is still useful: it means the result is less sensitive to client-level leakage.

The baseline is evaluated under both designs as well, so the model is never compared against a different test population.

## 3. Leakage audit

A feature is safe only if it would be known at the moment the review queue is generated.

| Feature | Decision-time source | Leakage risk | Audit result |
|---|---|---|---|
| `feb_impressions` | February search performance | Low | Keep |
| `feb_clicks` | February search performance | Low | Keep |
| `feb_ctr` | February clicks / impressions | Low | Keep |
| `feb_position` | February search performance | Low | Keep |
| `content_age_days` | Page age at decision time | Low | Keep |
| `march_clicks` | Future outcome window | **High if used as a feature** | Target only |
| `went_dark` | Derived from March clicks | **High if used as a feature** | Target only |
| `client_hash_id` | Entity/group identifier | Can create identity memorisation | Grouping only, not a feature |

The main leakage boundary is therefore the **February → March** boundary. The model must not use March clicks, March CTR, March position, March sessions, or any other post-decision signal as an input.

In [ ]:
# Programmatic leakage guard.
forbidden_feature_terms = [
    "march", "outcome", "went_dark", "future", "label", "target"
]

feature_lower = [c.lower() for c in feature_cols]
violations = [
    c for c in feature_lower
    if any(term in c for term in forbidden_feature_terms)
]

print("Model features:", feature_cols)
print("Forbidden feature-name matches:", violations)

assert not violations, (
    "Potential leakage detected in feature names: "
    + str(violations)
)

assert "march_clicks" not in feature_cols
assert "went_dark" not in feature_cols

print("Leakage guard passed: future outcome fields are not model features.")

In [ ]:
# Check that the baseline also uses only decision-time fields.
baseline_fields = [
    "content_age_days",
    "feb_ctr",
    "feb_position",
]

print("Baseline fields:", baseline_fields)
assert all(c in feature_cols for c in baseline_fields)
assert not any("march" in c.lower() for c in baseline_fields)
print("Baseline leakage guard passed.")

## 4. Claim rewrite

The Week-5 model supports a narrower claim than “the model predicts which pages should be refreshed.”

### Claim that goes too far

> “The Logistic Regression predicts which pages need a refresh.”

### Evidence-safe rewrite

> “On the measured sample, the Logistic Regression ranked pages by their observed probability of going dark in the following month. Under grouped-by-client validation, it provides decision-support for prioritising pages for human refresh review. This is an association with a proxy outcome, not evidence that the model identifies pages that causally need refreshing.”

### Another claim that goes too far

> “Older content causes pages to go dark.”

### Evidence-safe rewrite

> “Content age was included as a decision-time feature, and the fitted model can show a directional association between age and the measured outcome in this sample. The analysis does not establish that age itself causes a page to lose clicks.”

### Validation claim

> “The model works well.”

### Evidence-safe rewrite

> “The model's measured Precision@20 and ROC-AUC can be reported under the grouped validation design. Those metrics describe performance on this held-out sample and should not be treated as a guarantee for unseen clients or future months.”

These rewrites deliberately use **observed, measured, directional, and decision-support** language and avoid causal or universal claims.

## 5. Self-check

- [x] Two research-paper findings are named.
- [x] Each finding has a constructive methodology question about labels/outcomes or validation/comparability.
- [x] The Week-5 model is re-run with an explicit grouped-by-client split.
- [x] Before/after validation designs are shown with the same model, target, baseline, and primary metric.
- [x] Leakage is audited feature by feature.
- [x] Future March outcome fields are explicitly blocked from the feature set.
- [x] Real held-out false-positive/false-negative examples can be inspected from the same model design.
- [x] Claims are rewritten using measured, observed, directional, and decision-support language.
- [ ] Run **Runtime → Run all** and confirm the repository's actual data path executes successfully.
- [ ] Save the executed notebook as `work/notebooks/w06_validation_audit.ipynb`.
- [ ] Commit the executed notebook to the repo.

### Final interpretation

The purpose of this notebook is not to make the model look stronger. It is to determine whether the Week-5 result survives a more honest validation design and whether the claims stay within what the data actually measured.